In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents.base import Document
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv
import os
from euriai.langchain import create_chat_model
import time
from euriai.langchain import EuriaiEmbeddings

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))
api_key = os.getenv("key") 

loader = DirectoryLoader("./data", glob="**/*.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(docs)


chat_model = create_chat_model(api_key=api_key, model="gpt-4o-mini", temperature=0.7)
model = chat_model

embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

db = Chroma.from_documents(chunks, embeddings)
retriever = db.as_retriever()

c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model.invoke("hello world")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 11, 'completion_tokens': 9, 'total_tokens': 20}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': None, 'finish_reason': 'stop', 'model': 'gpt-4o-mini', 'created': 1773336078}, id='lc_run--019ce311-2620-7bf0-855f-89050376299d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 9, 'total_tokens': 20})

In [3]:
def format_docs(docs: list[Document]):
    return "\n".join(doc.page_content for doc in docs)

In [4]:
pipeline = (
RunnablePassthrough.assign(answer=lambda x: x["query"].upper())
)

print(pipeline.invoke({"query": "hello"}))


{'query': 'hello', 'answer': 'HELLO'}


In [5]:
from langchain_core.runnables import RunnableLambda

template = """Answer the question based only on the following context:
{context}

Question: {question} 
"""
prompt = ChatPromptTemplate.from_template(template)

format_context = RunnableLambda(lambda x: format_docs(x["context"]))

rag_chain_from_docs = (
    RunnablePassthrough.assign(context=format_context)
    | prompt
    | model
    | StrOutputParser()
)

rag_chain_with_source = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

In [6]:
result = rag_chain_with_source.invoke(input="Who is the owner of the restaurant")

In [7]:
result

{'context': [Document(id='8d059fe4-b91e-49fb-8ea3-ca49bed092f7', metadata={'source': 'data\\founder.txt'}, page_content='Creating Chef Amico’s Restaurant'),
  Document(id='424c25dd-9abc-4405-a614-67caa77126ed', metadata={'source': 'data\\founder.txt'}, page_content='craft. His spirit of generosity and passion for food extends beyond the restaurant’s walls. He mentors young chefs,'),
  Document(id='847e63ee-3bd4-4cba-bf1b-939f508b7c3b', metadata={'source': 'data\\restaurant.txt'}, page_content="into Chef Amico. Her mission was to uncover the secret behind the restaurant's growing fame. She was greeted by Amico"),
  Document(id='782ce337-470c-4d4e-8434-7595acaf5054', metadata={'source': 'data\\founder.txt'}, page_content='and relish life’s simple pleasures. His restaurant was a haven where strangers became friends over plates of arancini')],
 'question': 'Who is the owner of the restaurant',
 'answer': 'The owner of the restaurant is Chef Amico.'}

### Why is that approach bad?

You will always retrieve top-k documents and pass them to the model. 
No matter how relevant the documents are

In [8]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4070.78it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
docs = result["context"]
contents = [doc.page_content for doc in docs]

In [10]:
contents

['Creating Chef Amico’s Restaurant',
 'craft. His spirit of generosity and passion for food extends beyond the restaurant’s walls. He mentors young chefs,',
 "into Chef Amico. Her mission was to uncover the secret behind the restaurant's growing fame. She was greeted by Amico",
 'and relish life’s simple pleasures. His restaurant was a haven where strangers became friends over plates of arancini']

In [11]:
pairs = []
for text in contents:
    pairs.append(["Who is the owner of the restaurant", text])

In [12]:
pairs

[['Who is the owner of the restaurant', 'Creating Chef Amico’s Restaurant'],
 ['Who is the owner of the restaurant',
  'craft. His spirit of generosity and passion for food extends beyond the restaurant’s walls. He mentors young chefs,'],
 ['Who is the owner of the restaurant',
  "into Chef Amico. Her mission was to uncover the secret behind the restaurant's growing fame. She was greeted by Amico"],
 ['Who is the owner of the restaurant',
  'and relish life’s simple pleasures. His restaurant was a haven where strangers became friends over plates of arancini']]

In [13]:
scores = cross_encoder.predict(pairs)
scores

array([-3.4635892, -3.683052 , -3.478982 , -5.3002462], dtype=float32)

[-3.4635891914367676,
 -3.6830520629882812,
 -3.4789819717407227,
 -5.300246238708496]

In [48]:
dict(zip(list(scores), docs))

{np.float32(-3.4635892): Document(id='24b9b425-ab6b-4da5-927b-5871a5357a48', metadata={'source': 'data\\founder.txt'}, page_content='Creating Chef Amico’s Restaurant'),
 np.float32(-3.683052): Document(id='21300d0a-0c6e-4c64-8b01-ea5c66d54408', metadata={'source': 'data\\founder.txt'}, page_content='craft. His spirit of generosity and passion for food extends beyond the restaurant’s walls. He mentors young chefs,')}

In [15]:
scored_docs = dict(zip([float(j) for j in scores], docs))
sorted_docs = sorted(scored_docs.items(), key=lambda x: x[0], reverse=True)
sorted_docs

[(-3.4635891914367676,
  Document(id='8d059fe4-b91e-49fb-8ea3-ca49bed092f7', metadata={'source': 'data\\founder.txt'}, page_content='Creating Chef Amico’s Restaurant')),
 (-3.4789819717407227,
  Document(id='847e63ee-3bd4-4cba-bf1b-939f508b7c3b', metadata={'source': 'data\\restaurant.txt'}, page_content="into Chef Amico. Her mission was to uncover the secret behind the restaurant's growing fame. She was greeted by Amico")),
 (-3.6830520629882812,
  Document(id='424c25dd-9abc-4405-a614-67caa77126ed', metadata={'source': 'data\\founder.txt'}, page_content='craft. His spirit of generosity and passion for food extends beyond the restaurant’s walls. He mentors young chefs,')),
 (-5.300246238708496,
  Document(id='782ce337-470c-4d4e-8434-7595acaf5054', metadata={'source': 'data\\founder.txt'}, page_content='and relish life’s simple pleasures. His restaurant was a haven where strangers became friends over plates of arancini'))]

In [16]:
reranked_docs = [doc for _, doc in sorted_docs][0:2]
reranked_docs

[Document(id='8d059fe4-b91e-49fb-8ea3-ca49bed092f7', metadata={'source': 'data\\founder.txt'}, page_content='Creating Chef Amico’s Restaurant'),
 Document(id='847e63ee-3bd4-4cba-bf1b-939f508b7c3b', metadata={'source': 'data\\restaurant.txt'}, page_content="into Chef Amico. Her mission was to uncover the secret behind the restaurant's growing fame. She was greeted by Amico")]

### Integrate that in LCEL

In [17]:
# retriever which retrieves more than 4 documents
retriever = db.as_retriever(search_kwargs={"k": 10})

In [18]:
from sentence_transformers import CrossEncoder
from langchain_core.runnables import RunnableLambda

def rerank_documents(input_data):
    query = input_data["question"]
    docs = input_data["context"]

    cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    contents = [doc.page_content for doc in docs]

    pairs = [(query, text) for text in contents]
    scores = cross_encoder.predict(pairs)

    scored_docs = zip(scores, docs)
    sorted_docs = sorted(scored_docs, key=lambda x: x[0], reverse=True)
    return [doc for _, doc in sorted_docs]


template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
model = model

rag_chain_from_docs = (
    RunnablePassthrough.assign(context=RunnableLambda(rerank_documents))
    | prompt
    | model
    | StrOutputParser()
)

rag_chain_with_source = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

In [19]:
result = rag_chain_with_source.invoke(input="Who is the owner of the restaurant")
result

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4226.10it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'context': [Document(id='8d059fe4-b91e-49fb-8ea3-ca49bed092f7', metadata={'source': 'data\\founder.txt'}, page_content='Creating Chef Amico’s Restaurant'),
  Document(id='424c25dd-9abc-4405-a614-67caa77126ed', metadata={'source': 'data\\founder.txt'}, page_content='craft. His spirit of generosity and passion for food extends beyond the restaurant’s walls. He mentors young chefs,'),
  Document(id='847e63ee-3bd4-4cba-bf1b-939f508b7c3b', metadata={'source': 'data\\restaurant.txt'}, page_content="into Chef Amico. Her mission was to uncover the secret behind the restaurant's growing fame. She was greeted by Amico"),
  Document(id='782ce337-470c-4d4e-8434-7595acaf5054', metadata={'source': 'data\\founder.txt'}, page_content='and relish life’s simple pleasures. His restaurant was a haven where strangers became friends over plates of arancini'),
  Document(id='50a1e391-9dc4-4a7a-9ca4-24b94b13ae06', metadata={'source': 'data\\restaurant.txt'}, page_content='Leaving the restaurant, Elena knew h

### LLM-based Document Compressor

In [20]:
from langchain_core.prompts import PromptTemplate

DOCUMENT_EVALUATOR_PROMPT = PromptTemplate(
    input_variables=["document", "question"],
    template="""You are an AI language model assistant. Your task is to evaluate the provided document to determine if it is suited to answer the given user question. 
    Assess the document for its relevance to the question, the completeness of information, and the accuracy of the content.

    Original question: {question}
    Document for Evaluation: {document}
    Evaluation Result: <<'True' if the document is suited to answer the question, 'False' if it is not>>

    Note: Conclude with a 'True' or 'False' based on your analysis of the document's relevance, completeness, and accuracy in relation to the question.""",
)

In [22]:
from langchain_core.documents.base import Document

documents = [
    Document(page_content="The owner is Guivanni"),
    Document(page_content="Pizza Salami costs 10$"),
    Document(page_content="We close the restaurant at 10p.m each day"),
]

model = chat_model
compression_chain = DOCUMENT_EVALUATOR_PROMPT | model | StrOutputParser()

In [24]:
compression_chain.invoke(
    {"question": "Who is the owner of the restaurant", "document": documents[0]}
)

'True'

### Now lets make that dynamic

In [25]:
def evaluate_documents(input: dict):
    documents = input.get("documents", [])
    question = input.get("question")

    DOCUMENT_EVALUATOR_PROMPT = PromptTemplate(
        input_variables=["document", "question"],
        template="""You are an AI language model assistant. Your task is to evaluate the provided document to determine if it is suited to answer the given user question. Assess the document for its relevance to the question, the completeness of information, and the accuracy of the content.

        Original question: {question}
        Document for Evaluation: {document}
        Evaluation Result: <<'True' if the document is suited to answer the question, 'False' if it is not>>

        Note: Conclude with a 'True' or 'False' based on your analysis of the document's relevance, completeness, and accuracy in relation to the question.""",
    )
    model = chat_model
    compression_chain = DOCUMENT_EVALUATOR_PROMPT | model | StrOutputParser()

    results = []
    for document in documents:
        evaluation_result = compression_chain.invoke(
            {"document": document.page_content, "question": question}
        )
        result = evaluation_result == "True"
        print(result)
        results.append(result)

    filtered_documents = [doc for doc, res in zip(documents, results) if res]

    return filtered_documents

In [26]:
_input = {
    "documents": [
        Document(page_content="The owner is Guivanni"),
        Document(page_content="Pizza Salami costs 10$"),
        Document(page_content="We close the restaurant at 10p.m each day"),
    ],
    "question": "Who is the owner of the restaurant?",
}

results = evaluate_documents(_input)
print(results)

True
False
False
[Document(metadata={}, page_content='The owner is Guivanni')]
